In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import sys
from pyspark.sql import SparkSession
print("done")

done


In [2]:
spark = SparkSession.builder \
 .master("local") \
 .appName("Project") \
 .getOrCreate()

In [3]:
pwd

'/home/jovyan'

In [4]:
#pollution dataset
pollution_df = spark.read.csv("data/global air pollution dataset.csv", header=True, inferSchema=True)
pollution_df.show(vertical=True)

-RECORD 0----------------------------------
 Country            | Russian Federation   
 City               | Praskoveya           
 AQI Value          | 51                   
 AQI Category       | Moderate             
 CO AQI Value       | 1                    
 CO AQI Category    | Good                 
 Ozone AQI Value    | 36                   
 Ozone AQI Category | Good                 
 NO2 AQI Value      | 0                    
 NO2 AQI Category   | Good                 
 PM2.5 AQI Value    | 51                   
 PM2.5 AQI Category | Moderate             
-RECORD 1----------------------------------
 Country            | Brazil               
 City               | Presidente Dutra     
 AQI Value          | 41                   
 AQI Category       | Good                 
 CO AQI Value       | 1                    
 CO AQI Category    | Good                 
 Ozone AQI Value    | 5                    
 Ozone AQI Category | Good                 
 NO2 AQI Value      | 1         

In [5]:
pollution_df.printSchema()

root
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- AQI Value: integer (nullable = true)
 |-- AQI Category: string (nullable = true)
 |-- CO AQI Value: integer (nullable = true)
 |-- CO AQI Category: string (nullable = true)
 |-- Ozone AQI Value: integer (nullable = true)
 |-- Ozone AQI Category: string (nullable = true)
 |-- NO2 AQI Value: integer (nullable = true)
 |-- NO2 AQI Category: string (nullable = true)
 |-- PM2.5 AQI Value: integer (nullable = true)
 |-- PM2.5 AQI Category: string (nullable = true)



In [6]:
#GDP per capita data to combine
GDP_df = spark.read.csv("data/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_245.csv", header=True, inferSchema=True)
GDP_2022_df = GDP_df.select("Country Name", "Country Code", GDP_df["2022"])

In [7]:
from pyspark.sql.functions import col

missing_2022_df = GDP_df.filter(col("2022").isNull()) \
                       .select("Country Name", "Country Code")

missing_2022_df.show()

+--------------------+------------+
|        Country Name|Country Code|
+--------------------+------------+
|                Cuba|         CUB|
|             Eritrea|         ERI|
|           Gibraltar|         GIB|
|      Not classified|         INX|
|St. Martin (Frenc...|         MAF|
|Korea, Dem. Peopl...|         PRK|
|         South Sudan|         SSD|
|British Virgin Is...|         VGB|
|         Yemen, Rep.|         YEM|
+--------------------+------------+



In [8]:
# Identify countries that don't have a matching name in both datasets
non_matching_countries = pollution_df.select(col("Country").alias("country_name")).distinct().join(GDP_2022_df.select(col("Country Name").alias("country_name")).distinct(),on="country_name",how="left_anti")
non_matching_countries.show(n=non_matching_countries.count(), truncate=False)


+----------------------------------------------------+
|country_name                                        |
+----------------------------------------------------+
|Côte d'Ivoire                                       |
|Yemen                                               |
|State of Palestine                                  |
|Republic of Korea                                   |
|Turkey                                              |
|Kingdom of Eswatini                                 |
|Republic of North Macedonia                         |
|Slovakia                                            |
|Somalia                                             |
|United Republic of Tanzania                         |
|Bolivia (Plurinational State of)                    |
|Venezuela (Bolivarian Republic of)                  |
|Congo                                               |
|Democratic Republic of the Congo                    |
|Saint Kitts and Nevis                               |
|United St

In [9]:
!pip install pycountry
import pycountry
from pyspark.sql import Row

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 27.9 MB/s eta 0:00:0000:0100:01


In [10]:
# use pycountry to get ISO3 for each country to avoid dealing with masmatched names
def get_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except:
        return None

countries = pollution_df.select("country").distinct().collect()

mapping_data = []
for row in countries:
    country_name = row["country"]
    iso3 = get_iso3(country_name)
    mapping_data.append(Row(iso3=iso3, country=country_name))

mapping_df = spark.createDataFrame(mapping_data)
mapping_df.show(mapping_df.count(), truncate=False)
print(mapping_df.count())

+----+----------------------------------------------------+
|iso3|country                                             |
+----+----------------------------------------------------+
|CIV |Côte d'Ivoire                                       |
|TCD |Chad                                                |
|PRY |Paraguay                                            |
|YEM |Yemen                                               |
|NULL|State of Palestine                                  |
|SEN |Senegal                                             |
|SWE |Sweden                                              |
|CPV |Cabo Verde                                          |
|NULL|Republic of Korea                                   |
|GUY |Guyana                                              |
|PHL |Philippines                                         |
|ERI |Eritrea                                             |
|MYS |Malaysia                                            |
|SGP |Singapore                         

In [11]:
mapping_df.filter("iso3 IS NULL").show(truncate=False)

+----+----------------------------------+
|iso3|country                           |
+----+----------------------------------+
|NULL|State of Palestine                |
|NULL|Republic of Korea                 |
|NULL|Turkey                            |
|NULL|Bolivia (Plurinational State of)  |
|NULL|Venezuela (Bolivarian Republic of)|
|NULL|Democratic Republic of the Congo  |
|NULL|Iran (Islamic Republic of)        |
|NULL|NULL                              |
+----+----------------------------------+



In [12]:
from pyspark.sql.functions import when
mapping_df = mapping_df.withColumn(
    "iso3",
    when(col("country") == "State of Palestine", "PSE")
    .when(col("country") == "Republic of Korea", "KOR")
    .when(col("country") == "Turkey", "TUR")
    .when(col("country") == "Bolivia (Plurinational State of)", "BOL")
    .when(col("country") == "Venezuela (Bolivarian Republic of)", "VEN")
    .when(col("country") == "Democratic Republic of the Congo", "COD")
    .when(col("country") == "Congo", "COG")
    .when(col("country") == "Iran (Islamic Republic of)", "IRN")
    .otherwise(col("iso3"))
)

In [13]:
pollution_df = pollution_df.join(mapping_df, on="country", how="left")

In [14]:
mapping_df.filter("iso3 IS NULL").show(truncate=False)

+----+-------+
|iso3|country|
+----+-------+
|NULL|NULL   |
+----+-------+



In [15]:
pollution_df.show()

+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+----+
|             Country|            City|AQI Value|        AQI Category|CO AQI Value|CO AQI Category|Ozone AQI Value|  Ozone AQI Category|NO2 AQI Value|NO2 AQI Category|PM2.5 AQI Value|  PM2.5 AQI Category|iso3|
+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+----+
|             Belgium|           Puurs|       64|            Moderate|           1|           Good|             29|                Good|            7|            Good|             64|            Moderate| BEL|
|              Brazil|Presidente Dutra|       41|                Good|           1|           Good|              5|                Good|            1|          

In [28]:
#add GDP per capita column
from pyspark.sql.functions import coalesce

GDP_2022_df = GDP_df.select(
    col("Country Code").alias("iso3"),
    coalesce(
        col("2022"),
        col("2021"),
        col("2023"),
        col("2020"),
        col("2024"),
        col("2019"),
        col("2018"),
        col("2017"),
        col("2016"),
        col("2015"),
        col("2014"),
        col("2013"),
        col("2012"),
        col("2011")
    ).alias("gdp_per_capita_2022")
)

In [29]:
final_df = pollution_df.join(
    GDP_2022_df,
    on="iso3",
    how="left"
)
final_df.show()

+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+
|iso3|             Country|            City|AQI Value|        AQI Category|CO AQI Value|CO AQI Category|Ozone AQI Value|  Ozone AQI Category|NO2 AQI Value|NO2 AQI Category|PM2.5 AQI Value|  PM2.5 AQI Category|gdp_per_capita_2022|
+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+
| BEL|             Belgium|           Puurs|       64|            Moderate|           1|           Good|             29|                Good|            7|            Good|             64|            Moderate|   50605.7496677086|
| BRA|              Brazil|Presidente Dutra|       41|                Good|     

In [31]:
#add GDP column
df = spark.read.csv("data/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_126992.csv", header=True, inferSchema=True)

join_df = df.select(
    col("Country Code").alias("iso3"),
    coalesce(
        col("2022"),
        col("2021"),
        col("2023"),
        col("2020"),
        col("2024"),
        col("2019"),
        col("2018"),
        col("2017"),
        col("2016"),
        col("2015"),
        col("2014"),
        col("2013"),
        col("2012"),
        col("2011")
    ).alias("gdp_2022")
)

In [32]:
final_df = final_df.join(
    join_df,
    on="iso3",
    how="left"
)
final_df.show()

+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+-------------------+
|iso3|             Country|            City|AQI Value|        AQI Category|CO AQI Value|CO AQI Category|Ozone AQI Value|  Ozone AQI Category|NO2 AQI Value|NO2 AQI Category|PM2.5 AQI Value|  PM2.5 AQI Category|gdp_per_capita_2022|           gdp_2022|
+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+-------------------+
| BEL|             Belgium|           Puurs|       64|            Moderate|           1|           Good|             29|                Good|            7|            Good|             64|            Moderate|   50605.7496677086|5.91085783326267E11|


In [34]:
#add column for urban_population_2022
df = spark.read.csv("data/API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_121583.csv", header=True, inferSchema=True)

join_df = df.select(
    col("Country Code").alias("iso3"),
    coalesce(
        col("2022"),
        col("2021"),
        col("2023"),
        col("2020"),
        col("2024"),
        col("2019"),
        col("2018"),
        col("2017"),
        col("2016"),
        col("2015"),
        col("2014"),
        col("2013"),
        col("2012"),
        col("2011")
    ).alias("urban_population_2022")
)

In [35]:
final_df = final_df.join(
    join_df,
    on="iso3",
    how="left"
)
final_df.show()

+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+-------------------+---------------------+
|iso3|             Country|            City|AQI Value|        AQI Category|CO AQI Value|CO AQI Category|Ozone AQI Value|  Ozone AQI Category|NO2 AQI Value|NO2 AQI Category|PM2.5 AQI Value|  PM2.5 AQI Category|gdp_per_capita_2022|           gdp_2022|urban_population_2022|
+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+-------------------+---------------------+
| BEL|             Belgium|           Puurs|       64|            Moderate|           1|           Good|             29|                Good|            7|            Good|            

In [37]:
# add population_density_2022,pollution_mortality_2019
df = spark.read.csv("data/API_EN.POP.DNST_DS2_en_csv_v2_1453.csv", header=True, inferSchema=True)

join_df = df.select(
    col("Country Code").alias("iso3"),
    coalesce(
        col("2022"),
        col("2021"),
        col("2023"),
        col("2020"),
        col("2024"),
        col("2019"),
        col("2018"),
        col("2017"),
        col("2016"),
        col("2015"),
        col("2014"),
        col("2013"),
        col("2012"),
        col("2011")
    ).alias("population_density_2022")
)

In [38]:
final_df = final_df.join(
    join_df,
    on="iso3",
    how="left"
)
final_df.show()

+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+-------------------+---------------------+-----------------------+
|iso3|             Country|            City|AQI Value|        AQI Category|CO AQI Value|CO AQI Category|Ozone AQI Value|  Ozone AQI Category|NO2 AQI Value|NO2 AQI Category|PM2.5 AQI Value|  PM2.5 AQI Category|gdp_per_capita_2022|           gdp_2022|urban_population_2022|population_density_2022|
+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+-------------------+---------------------+-----------------------+
| BEL|             Belgium|           Puurs|       64|            Moderate|           1|           Good|        

In [39]:
# add pollution_mortality_2019 column
df = spark.read.csv("data/API_SH.STA.AIRP.P5_DS2_en_csv_v2_9827.csv", header=True, inferSchema=True)

join_df = df.select(
    col("Country Code").alias("iso3"),
    coalesce(
        col("2022"),
        col("2021"),
        col("2023"),
        col("2020"),
        col("2024"),
        col("2019"),
        col("2018"),
        col("2017"),
        col("2016"),
        col("2015"),
        col("2014"),
        col("2013"),
        col("2012"),
        col("2011")
    ).alias("pollution_mortality_2019")
)

In [40]:
final_df = final_df.join(
    join_df,
    on="iso3",
    how="left"
)
final_df.show()

+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+-------------------+---------------------+-----------------------+------------------------+
|iso3|             Country|            City|AQI Value|        AQI Category|CO AQI Value|CO AQI Category|Ozone AQI Value|  Ozone AQI Category|NO2 AQI Value|NO2 AQI Category|PM2.5 AQI Value|  PM2.5 AQI Category|gdp_per_capita_2022|           gdp_2022|urban_population_2022|population_density_2022|pollution_mortality_2019|
+----+--------------------+----------------+---------+--------------------+------------+---------------+---------------+--------------------+-------------+----------------+---------------+--------------------+-------------------+-------------------+---------------------+-----------------------+------------------------+
| BEL|             Belgium|          

In [ ]:
#save final dataset
final_df.write.csv("output/final_df", header=True)